# data collection

Runs the pipeline, returns the three model tables:

| Table | Key | Contents |
|---|---|---|
| `bond_features` | bond | terms, AGG market/par value, renormalised weight `agg_weight` (sums to 1) |
| `prices` | bond, month-end date | clean/accrued/dirty price, `spread_bps`, `yield_pct`, `transaction_cost_frac`, `is_stale` |
| `yield_curve` | date | zero yields at the 8 key rates |

Settings (dates, tables, thresholds, assumptions) are all in `config.py`. Needs WRDS access (`pip install wrds`).
Set `use_cache=True` to reload the last WRDS pull from `data/raw`, and `download=False` to reuse the saved AGG file.

## TODO:

**Sample and coverage**
- [ ] **Sample window.** Prices run Jan 2025 to Dec 2025 (`END_DATE` in `config.py`) because CRSP Treasury quotes on WRDS stop at 2025-12-31. The bond universe, though, is the AGG holdings of Nov 2026. Are we fine using today's holdings with last year's prices, or should we use holdings from around Dec 2025?
- [ ] **BondRet bonds have no Dec 2025 row.** BondRet ends Nov 2025, so about 6,600 corporates stop one month early. Options: carry the last price forward and mark it stale, fill December from TRACE trades, or end the sample at Nov 2025.
- [ ] **MBS pools are only priced Jan–Feb 2025.** Enhanced TRACE MBS data (`trace.trace_spds_mbs_enhanced`) ends 2025-03-17, and the newer standard table has no CUSIPs. Options: drop MBS from the model, carry Feb prices forward, or find another source for MBS prices.
- [ ] **28 TBA positions** (CUSIPs starting `01F` or `21H`) have no prices. Exclude them, or map each one to a representative pool?
- [ ] **`MAX_STALE_MONTHS` is `None`**, so stale prices are carried forward with no limit. Choose a cap.

**Assumptions to verify (marked VERIFY / PLACEHOLDER / ASSUMED in `config.py`)**
- [ ] `HALF_SPREAD_BPS`: placeholder transaction costs, used wherever TRACE has no observed bid/ask.
- [ ] `MIN_DENOM`: minimum trade sizes are assumed.
- [ ] `BONDRET_SPREAD_TO_BPS`: we assume BondRet `t_spread` is in percentage points. Check this against the WRDS docs.
- [ ] `CRSP_EXCLUDE_ITYPES = (11, 12)`: confirm these are the TIPS issue types.
- [ ] `SECTOR_RULES`: check the iShares sector → model sector mapping against the table printed in step 1.
- [ ] `MBS_CPR = 0`: pass-through cash flows assume no prepayment, which overstates MBS duration.
- [ ] The Treasury auctions feed (`features.fetch_treasury_auctions`): verify the TreasuryDirect field names.

**Known modelling gaps**
- [ ] Callable corporates are valued as bullet bonds, so their solved yields can differ from the iShares yield-to-worst. This shows up as the −6 bps mean gap for financials in `checks()`. Do we need call schedules?


In [10]:
from data_files.pipeline import run, checks

tables = run(use_cache=False, download=True)

1) AGG universe
  holdings weight in file: 81.8%, kept after CUSIP filter: 84.0%
cusip                string[python]
name                         object
ishares_sector               object
asset_class                  object
agg_market_value            float64
agg_weight_pct              float64
agg_par_value               float64
agg_maturity_date    datetime64[ns]
agg_coupon_pct              float64
agg_duration                float64
agg_mod_duration            float64
agg_ytm_pct                 float64
agg_weight_raw              float64
cusip8               string[python]
model_sector                 object
dtype: object
                                            n  weight_pct
ishares_sector          model_sector                     
ABS                     (excluded)        149        0.23
Agency                  (excluded)        153        0.32
CMBS                    (excluded)        383        0.96
Cash and/or Derivatives (excluded)          2        2.61
Financial Institu

/Users/rosana/CODE/Capstone/data_cleaning/data_files/prices.py:107: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p["is_stale_prev"] = p.groupby("cusip")["is_stale"].shift(1).fillna(True).astype(bool)


Prices table dtypes:
cusip                            string[python]
month_end_date                   datetime64[ns]
price_date                       datetime64[ns]
clean_price                             Float64
accrued_interest_crsp                   Float64
half_spread_frac                        Float64
source                                   object
yield_pct_reported                      Float64
spread_bps_reported                     Float64
volume                                  Float64
is_stale                                   bool
months_since_observed                     int32
valuation_date                   datetime64[ns]
accrued_interest_diff_vs_crsp           float64
accrued_interest                        float64
dirty_price                             Float64
spread_source                            object
yield_source                             object
spread_bps                              Float64
yield_pct                               Float64
is_stale_prev      

In [3]:
checks(tables)

PASS  weights sum to 1
PASS  unique CUSIPs
PASS  one row per (cusip, month-end)
PASS  no missing clean prices
PASS  dirty >= clean
PASS  all 8 key rates present

Solved yield minus AGG-file YTM (bps), both at the holdings date 2026-11-09 (tests the yield maths, not the data):
               count  mean  50%
model_sector                   
financial     2369.0  -6.3  0.5
industrial    5106.0   0.3  0.5
treasury       293.0   0.7  0.6
utility       1466.0  -1.7  0.4

Sample: 2025-01-31 to 2025-12-31 (END_DATE 2025-12-31); universe = AGG holdings of 2026-11-09
PASS  prices reach the END_DATE month-end (2025-12-31)
Last priced month-end by source: {'bondret': datetime.date(2025, 11, 30), 'crsp': datetime.date(2025, 12, 31), 'trace_corp': datetime.date(2025, 12, 31), 'trace_mbs': datetime.date(2025, 2, 28)}

Stale share by source:
 source
bondret       0.011
crsp          0.000
trace_corp    0.026
trace_mbs     0.111
Name: is_stale, dtype: float64


In [9]:
tables["bond_features"].head(15)

,cusip,name,ishares_sector,asset_class,agg_market_value,agg_par_value,agg_duration,agg_mod_duration,agg_ytm_pct,agg_weight_raw,model_sector,issue_date,maturity_date,coupon_pct,coupon_frequency,day_count,is_amortising,min_denomination,terms_source,agg_weight
0,91282CMM0,TREASURY NOTE,Treasury,Fixed Income,5.620210e+08,563326000.0,6.72,6.72,4.82,0.0040,treasury,2025-02-15,2035-02-15,4.625,2,ACT/ACT,False,100.0,wrds,0.007282
1,91282CGQ8,TREASURY NOTE,Treasury,Fixed Income,5.532966e+08,559636000.0,3.03,3.03,4.63,0.0040,treasury,2023-02-28,2030-02-28,4.0,2,ACT/ACT,False,100.0,wrds,0.007282
2,91282CNC1,TREASURY NOTE,Treasury,Fixed Income,5.498705e+08,560863000.0,6.88,6.88,4.83,0.0040,treasury,2025-05-15,2035-05-15,4.25,2,ACT/ACT,False,100.0,wrds,0.007282
3,91282CLW9,TREASURY NOTE,Treasury,Fixed Income,5.436526e+08,552671000.0,6.55,6.55,4.81,0.0039,treasury,2024-11-15,2034-11-15,4.25,2,ACT/ACT,False,100.0,wrds,0.007100
4,91282CKQ3,TREASURY NOTE,Treasury,Fixed Income,5.423878e+08,544734000.0,6.18,6.18,4.78,0.0039,treasury,2024-05-15,2034-05-15,4.375,2,ACT/ACT,False,100.0,wrds,0.007100
5,91282CLF6,TREASURY NOTE,Treasury,Fixed Income,5.404743e+08,569018000.0,6.52,6.52,4.80,0.0039,treasury,2024-08-15,2034-08-15,3.875,2,ACT/ACT,False,100.0,wrds,0.007100
6,91282CJJ1,TREASURY NOTE,Treasury,Fixed Income,5.373463e+08,534025000.0,5.81,5.82,4.76,0.0039,treasury,2023-11-15,2033-11-15,4.5,2,ACT/ACT,False,100.0,wrds,0.007100
7,91282CNT4,TREASURY NOTE,Treasury,Fixed Income,5.359609e+08,553646000.0,7.12,7.13,4.84,0.0039,treasury,2025-08-15,2035-08-15,4.25,2,ACT/ACT,False,100.0,wrds,0.007100
8,91282CPJ4,TREASURY NOTE,Treasury,Fixed Income,5.300396e+08,553311000.0,7.27,7.27,4.85,0.0038,treasury,2025-11-15,2035-11-15,4.0,2,ACT/ACT,False,100.0,wrds,0.006918
9,91282CJZ5,TREASURY NOTE,Treasury,Fixed Income,5.180957e+08,538515000.0,6.14,6.14,4.78,0.0037,treasury,2024-02-15,2034-02-15,4.0,2,ACT/ACT,False,100.0,wrds,0.006736


In [8]:
tables["prices"].head(15)

,cusip,month_end_date,price_date,valuation_date,source,is_stale,is_stale_prev,usable_for_residual_variance,clean_price,accrued_interest,dirty_price,spread_bps,spread_source,yield_pct,yield_source,transaction_cost_frac,cost_source,volume,accrued_interest_diff_vs_crsp
0,00037BAC6,2025-01-31,2025-01-31,2025-01-31,bondret,False,True,False,88.68,1.008681,89.688681,57.292882,solved,5.464518,reported,0.0025,sector_default,690778.9877,NaN
1,00037BAC6,2025-02-28,2025-02-28,2025-02-28,bondret,False,False,True,90.503657,1.336806,91.840462,0.137848,reported,5.36782,reported,0.0025,sector_default,11597808.569425,NaN
2,00037BAC6,2025-03-31,2025-03-31,2025-03-31,bondret,False,False,True,88.87188,1.737847,90.609727,0.090538,reported,5.398924,reported,0.0025,sector_default,2455501.7928,NaN
3,00037BAC6,2025-04-30,2025-04-30,2025-04-30,bondret,False,False,True,87.04375,2.090278,89.134028,0.494399,reported,5.509578,reported,0.0025,sector_default,700440.82,NaN
4,00037BAC6,2025-05-31,2025-05-31,2025-05-31,bondret,False,False,True,85.824962,0.279514,86.104475,0.110639,reported,5.803865,reported,0.0025,sector_default,27099366.304,NaN
5,00037BAC6,2025-06-30,2025-06-30,2025-06-30,bondret,False,False,True,87.690293,0.631944,88.322238,0.124318,reported,5.571186,reported,0.0025,sector_default,14154277.45045,NaN
6,00037BAC6,2025-07-31,2025-07-31,2025-07-31,bondret,False,False,True,88.086,1.008681,89.094681,0.258912,reported,5.552271,reported,0.0025,sector_default,17824338.32319,NaN
7,00037BAC6,2025-08-31,2025-08-31,2025-08-31,bondret,False,False,True,88.443975,1.373264,89.817239,0.142934,reported,5.400437,reported,0.0025,sector_default,22491646.7461,NaN
8,00037BAC6,2025-09-30,2025-09-30,2025-09-30,bondret,False,False,True,90.165673,1.725694,91.891368,0.075684,reported,5.284652,reported,0.0025,sector_default,19977047.56852,NaN
9,00037BAC6,2025-10-31,2025-10-31,2025-10-31,bondret,False,False,True,90.808,2.102431,92.910431,0.00303,reported,5.217524,reported,0.0025,sector_default,21415984.6896,NaN


In [7]:
tables["yield_curve"]

,zero_yield_6m_pct,zero_yield_1y_pct,zero_yield_2y_pct,zero_yield_3y_pct,zero_yield_5y_pct,zero_yield_7y_pct,zero_yield_10y_pct,zero_yield_30y_pct,rmse_price,n_used,n_quoted
date,,,,,,,,,,,
2025-01-31,4.160720,4.161976,4.177834,4.207683,4.294675,4.399921,4.560591,4.842334,0.158401,365,396
2025-02-28,4.132105,4.055711,3.965345,3.933295,3.966970,4.059133,4.218710,4.589140,0.256878,374,396
2025-03-31,4.128449,4.003907,3.875268,3.844807,3.917824,4.051307,4.254801,4.721247,0.220510,379,396
2025-04-30,4.064548,3.831191,3.599557,3.553027,3.693330,3.921024,4.241509,4.865756,0.220363,380,396
2025-05-30,4.142136,4.028969,3.894274,3.850544,3.928811,4.114860,4.436148,5.002173,0.169797,356,397
2025-06-30,4.135313,3.933581,3.715772,3.654865,3.758423,3.970158,4.298315,4.927765,0.187220,380,398
2025-07-31,4.238872,4.119928,3.904765,3.831908,3.930586,4.128171,4.421585,5.050415,0.185887,381,397
2025-08-29,3.906708,3.802844,3.597721,3.536940,3.678350,3.927865,4.296065,5.158575,0.217591,358,399
2025-09-30,3.806924,3.693474,3.588545,3.585599,3.722865,3.924468,4.222011,4.901629,0.185553,385,400


Also in `tables`: `key_rate_changes_bps` (monthly curve changes for the covariance), `curve_params`, `dropped_bonds`, `trace_log`, `universe`, `auctions`. Everything is saved to `data/processed/*.csv`.